# 02 — Tách Silver và Rejected từ Bronze

### 1. Xem dữ liệu trước khi xử lý

Notebook 01 tạo bảng `delivery_bronze`. Hai cột thời gian đang là chuỗi
dạng `MM-DD HH:MM:SS`.

In [ ]:
import sys

sys.path.insert(0, "/home/jovyan/pipeline")
import steps

spark = steps.get_spark("Delivery - Silver")

In [ ]:
bronze = spark.read.format("delta").load(steps.BRONZE)
bronze.select("order_id", "accept_time", "delivery_time").show(5, truncate=False)

### 2. Áp dụng điều kiện Silver

1. Ép kiểu mã và tọa độ. Gắn năm 2022 vào thời gian nhận, giao.
2. Tính `delivery_minutes = (delivery_ts - accept_ts) / 60` với hiệu thời gian tính bằng giây.
3. Giữ đơn có hai timestamp hợp lệ và thời lượng dương. Đưa đơn còn lại vào Rejected
   với lý do `invalid_timestamp` hoặc `non_positive_duration`.
4. Gắn cờ GPS, qua ngày, thời lượng trên 24 giờ và khoảng cách trên 50 km.
   Hai ngưỡng 24 giờ và 50 km dùng để chọn tập học Spark ở notebook 04.

`steps.transform_silver()` gọi `split_silver()`, ghi hai bảng và đối soát
mỗi khóa Bronze xuất hiện đúng một lần trong hợp Silver và Rejected.

In [1]:
result = steps.transform_silver(spark)
result

{'bronze_rows': 206431, 'silver_rows': 206062, 'rejected_rows': 369}

### 3. Đối chiếu kết quả

Trừ hai timestamp trên một dòng Silver rồi so với thời lượng đã ghi.
Xem số đơn theo từng lý do trong Rejected.

In [ ]:
from math import isclose

silver = spark.read.format("delta").load(steps.SILVER)
rejected = spark.read.format("delta").load(steps.REJECTED)
example = silver.select("order_id", "accept_ts", "delivery_ts", "delivery_minutes").first()
hand_minutes = (example.delivery_ts - example.accept_ts).total_seconds() / 60
assert isclose(hand_minutes, example.delivery_minutes, rel_tol=0, abs_tol=1e-9)
print(f"Đơn {example.order_id}  {example.delivery_ts} - {example.accept_ts}")
print(f"Tính tay {hand_minutes:.2f} phút, cột Silver {example.delivery_minutes:.2f} phút")
rejected.groupBy("reject_reason").count().show(truncate=False)